### **Setup**

In [ ]:
! git clone https://github.com/vllm-project/speculators.git

In [ ]:
%cd speculators

In [ ]:
# Speculators venv (for data prep and training)
! uv venv speculators_venv
! source speculators_venv/bin/activate
! uv pip install "vllm>=0.22.0"
! uv pip install -e .

In [ ]:
! pip uninstall -y torchaudio
! pip install -Uq datasets --break-system-packages

### **Generate responses**

In [ ]:
! ./scripts/response_regeneration/run_all.sh \
  --model "unsloth/Llama-3.2-3B-Instruct" \
  --dataset open-perfectblend \
  --limit 100000 \
  --max-tokens 2048 \
  --concurrency 256

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_WRITE_YOSEFW")


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(
    "yosefw/open-perfectblend-llama-3.2-3b-instruct",
    repo_type="dataset",
    exist_ok=True
)
api.upload_file(
    path_or_fileobj="./open-perfectblend_Llama-3.2-3B-Instruct.jsonl",
    path_in_repo="open-perfectblend_Llama-3.2-3B-Instruct.jsonl",
    repo_id="yosefw/open-perfectblend-llama-3.2-3b-instruct",
    repo_type="dataset",
)

### **Load Data**

In [ ]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="yosefw/open-perfectblend-llama-3.2-3b-instruct",
    filename="open-perfectblend_Llama-3.2-3B-Instruct.jsonl",
    repo_type="dataset",
    local_dir=".",          # drop it in the cwd instead of the cache
)

print(path)

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "json",
    data_files="open-perfectblend_Llama-3.2-3B-Instruct.jsonl",
    split="train",
)
print(ds)
print(ds[0]['metadata']['usage']['total_tokens'], ds[0]['metadata']['usage']['prompt_tokens'])

In [ ]:
ds_filtered = ds.filter(
    lambda row: row['metadata']['usage']['prompt_tokens'] <= 768,
    num_proc=4
)
ds_filtered.to_json(
    "filtered-open-perfectblend_Llama-3.2-3B-Instruct.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)
ds_filtered

### **Prepare data**

In [ ]:
! rm -r ./output

In [ ]:
# in speculators venv
! python scripts/prepare_data.py \
  --model "unsloth/Llama-3.2-3B-Instruct" \
  --data ./filtered-open-perfectblend_Llama-3.2-3B-Instruct.jsonl \
  --output ./output \
  --max-samples 100000 \
  --seq-length 1024 \
  --overwrite

### **Generate hidden states**

In [ ]:
import subprocess, json

log_file = open("/content/vllm.log", "w")
process = subprocess.Popen(
    [
        "python", "scripts/launch_vllm.py",
        "unsloth/Llama-3.2-3B-Instruct",
        "--gpu-memory-utilization", "0.25",
        "--max-model-len", "8200",
        "--target-layer-ids", "2", "8", "14", "20", "26",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,  # detaches from the notebook's process group so a cell interrupt (SIGINT) doesn't reach it
)

print(f"Started vLLM server with PID {process.pid}")

In [ ]:
import time, requests

for i in range(100):
    try:
        r = requests.get("http://127.0.0.1:8000/health")
        if r.status_code == 200:
            print("Server is ready!")
            break
    except requests.exceptions.ConnectionError:
        pass
    print(f"Waiting... ({i+1})")
    time.sleep(5)

In [ ]:
# import os, signal
# os.killpg(os.getpgid(process.pid), signal.SIGTERM)

### **Train DSpark**

In [ ]:
# in speculators venv
! CUDA_VISIBLE_DEVICES=0 torchrun --standalone --nproc_per_node 1 \
  scripts/train.py \
  --verifier-name-or-path "unsloth/Llama-3.2-3B-Instruct" \
  --data-path ./output \
  --save-path ./output/checkpoints \
  --draft-vocab-size 50000 \
  --epochs 3 \
  --train-data-ratio 0.96 \
  --total-seq-len 8192 \
  --max-anchors 1024 \
  --num-workers 12 \
  --speculator-type dspark \
  --num-layers 5 \
  --block-size 8 \
  --lr 3e-4 \
  --loss-fn '{"ce": 0.1, "tv": 0.9}' \
  --scheduler-type cosine \
  --scheduler-warmup-ratio 0.04 \
  --vllm-endpoint http://localhost:8000/v1 \
  --target-layer-ids 2 8 14 20 26 \
  --on-missing generate \
  --on-generate delete \
  --no-resume-from-checkpoint \
  --log-freq 200 2>&1 | tee /content/train.log

In [ ]:
import os
import signal
os.killpg(os.getpgid(process.pid), signal.SIGTERM)

In [ ]:
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get("HF_WRITE")

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(repo_id="rasyosef/Llama-3.2-3B-Instruct-DSpark", exist_ok=True)

api.upload_folder(
    folder_path="/content/speculators/output/checkpoints/checkpoint_best",
    repo_id="rasyosef/Llama-3.2-3B-Instruct-DSpark",
    repo_type="model",
)

### **Serve and Evaluate**

In [ ]:
%cd scripts/evaluate
! pip install -qr requirements.txt

In [ ]:
import subprocess, os

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

log_file = open("/content/vllm_serve.log", "w")
process = subprocess.Popen(
    [
        "vllm", "serve", "rasyosef/Llama-3.2-3B-Instruct-DSpark", #"./output/checkpoints/checkpoint_best",
        "--port", "8000",
        "--gpu-memory-utilization", "0.80"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,  # detaches from notebook's process group so cell interrupts don't kill it
)
print(f"Started vLLM server with PID {process.pid}")

import time, requests

for i in range(120):
    try:
        r = requests.get("http://127.0.0.1:8000/health")
        if r.status_code == 200:
            print("Server is ready!")
            break
    except requests.exceptions.ConnectionError:
        pass
    print(f"Waiting... ({i+1})")
    time.sleep(5)

In [ ]:
! python evaluate.py throughput --target http://localhost:8000/v1 --output-dir llama_dspark

In [ ]:
import pandas as pd

path = "llama_dspark/acceptance.csv"
df = pd.read_csv(path)

print(df.shape)
df.head(10)

In [ ]:
import os
import signal
os.killpg(os.getpgid(process.pid), signal.SIGTERM)